# 03 - Writing text and reading pictures

GPT-2 continues a story; CLIP puts a picture and a sentence in one space and classifies images
among labels it was never trained on. Both run on the CPU, in C#, and agree with transformers:
GPT-2's greedy output token for token, CLIP's logits to 1e-13.

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
#r "nuget: Gravicode.HFNet.GraviHub, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviTokenizers, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviTransformers, 0.7.0"

using Gravicode.HFNet.GraviTransformers;
using Gravicode.HFNet.GraviTransformers.Vision;

## 1. Continue a story

A decoder attends only to the tokens before each position, which is what lets it be asked what
comes next. The key/value cache makes each new token cost one row through the network.

In [ ]:
using var gpt = CausalLanguageModel.Load("gpt2");

// Greedy: deterministic, and exactly what transformers' generate(do_sample=False) returns.
gpt.Generate("The lighthouse keeper opened the door and", new GenerationSettings(MaxNewTokens: 30))

In [ ]:
// Sampling, streamed as it is written. The repetition penalty is transformers' rule.
var settings = new GenerationSettings(MaxNewTokens: 60, Sample: true, Temperature: 0.8, TopP: 0.95,
                                      RepetitionPenalty: 1.2, Seed: 7);
foreach (var piece in gpt.Stream("Once upon a time in a small harbour town,", settings))
    Console.Write(piece);

## 2. Labels it never saw

Each label becomes a sentence - `This is a photo of {label}.` - and the picture's similarity to each
sentence, times CLIP's learned temperature, goes through a softmax. No training.

In [ ]:
using Gravicode.HFNet.GraviHub;

using var clip = ClipModel.Load("openai/clip-vit-base-patch32");
var bee = Hub.DownloadDatasetFile("huggingface/documentation-images", "bee.jpg");

clip.ZeroShot(bee, ["a bee", "a flower", "a butterfly", "a bird", "a cat"])

The image is prepared exactly as `CLIPImageProcessor` prepares it: PIL's bicubic resize reproduced
to the byte, an integer centre crop, float32 normalisation. From a PNG the pixels are identical to
the reference's; a JPEG can differ by one level on some pixels, because PIL and ImageSharp decode
JPEG slightly differently.

In [ ]:
// Similarity between a picture and any sentence, in [-1, 1].
new[] { "a photo of an insect on a flower", "a city skyline at night" }
    .Select(text => (text, similarity: clip.Similarity(bee, text)))

## 3. Faster, a little less exact

Everything runs in double precision by default. The linear layers can run in float32 instead - the
same precision torch uses - for roughly 1.3-1.8x more speed.

In [ ]:
ComputeOptions.LinearLayers = Precision.Single;
var watch = System.Diagnostics.Stopwatch.StartNew();
var fast = gpt.Generate("The lighthouse keeper opened the door and", new GenerationSettings(MaxNewTokens: 30));
Console.WriteLine($"{watch.ElapsedMilliseconds} ms: {fast}");
ComputeOptions.LinearLayers = Precision.Double;